In [ ]:
import numpy as np
from utils import load_dataset , one_hot

In [ ]:
X_train,Y_train,X_test,Y_test=load_dataset()
print(X_train.shape, Y_train[0][4])
Y_train=one_hot(Y_train)
Y_train=Y_train.T
print(Y_train.shape, Y_train[:,1])



(1080, 64, 64, 3) 2
(6, 1080) [1. 0. 0. 0. 0. 0.]


parameters and hyperparameters

In [ ]:
def get_parameters():
    parameters={}


    parameters["Wc1"]=np.random.randn(3,3,3,32)*0.01
    parameters["bc1"]=np.zeros((1,1,1,32))
    parameters["Wc2"]=np.random.randn(3,3,32,64)*0.01
    parameters["bc2"]=np.zeros((1,1,1,64))
    parameters["Wfc1"]=np.random.randn(128,(16*16*64))*0.01
    parameters["bfc1"]=np.zeros((128,1))
    parameters["Wfc2"]=np.random.rand(6,128)*0.01
    parameters["bfc2"]=np.zeros((6,1))
    return parameters

def get_hparameters():
    hparameters={}

    hparameters["sc1"]=1
    hparameters["pc1"]=1
    hparameters["sc2"]=1
    hparameters["pc2"]=1
    hparameters["sp1"]=2
    hparameters["fp1"]=2
    hparameters["sp2"]=2
    hparameters["fp2"]=2
    return hparameters

p=get_parameters()
print(p.keys())

h=get_hparameters()
print(h.keys())

print(p["Wc1"].shape)
    




    



dict_keys(['Wc1', 'bc1', 'Wc2', 'bc2', 'Wfc1', 'bfc1', 'Wfc2', 'bfc2'])
dict_keys(['sc1', 'pc1', 'sc2', 'pc2', 'sp1', 'fp1', 'sp2', 'fp2'])
(3, 3, 3, 32)


padding

In [ ]:
def zero_pad(X,pad):
    X_pad=np.pad(X,((0,0),(pad,pad),(pad,pad),(0,0)))
    return X_pad


forward propagation

In [ ]:
def conv_forward(A_prev,W,b,s,p):

    (m,n_H_prev,n_W_prev,n_C_prev)=A_prev.shape
    (f,f,n_C_prev,n_C)=W.shape

    strides=s
    pad=p

    n_H= int (np.floor((n_H_prev+(2*pad)-f)/strides)+1)
    n_W= int (np.floor((n_W_prev+(2*pad)-f)/strides)+1)

    Z=np.zeros((m,n_H,n_W,n_C))
    A_prev_pad=zero_pad(A_prev,pad)

    W_flatten=W.reshape(-1,n_C)

    for h in range (0,n_H):
        for w in range(0,n_W):
            

            vert_start=(h*strides)
            vert_end=vert_start+f

            horz_start=(w*strides)
            horz_end=horz_start+f

            a_slice=A_prev_pad[:,vert_start:vert_end,horz_start:horz_end,:]
            a_slice_flat=a_slice.reshape(m,-1)
            Z[:,h,w,:]=(np.dot(a_slice_flat,W_flatten))+b.reshape(1,n_C)



    cache=(A_prev,W,b,Z,s,p)
    A=np.maximum(0,Z)

    return A,cache


In [ ]:
def pool_forward(A_prev,s,f):
    (m,n_H_prev,n_W_prev,n_C_prev)=A_prev.shape
    strides=s
    f=f

    n_H=int(1+((n_H_prev-f)/strides))
    n_W=int(1+((n_W_prev-f)/strides))
    A=np.zeros((m,n_H,n_W,n_C_prev))
    
    for h in range(0,n_H):
        for w in range(0,n_W):
        

            vert_start=(h*strides)
            vert_end=vert_start+f

            horz_start=(w*strides)
            horz_end=horz_start+f

            a_prev_slice=A_prev[:,vert_start:vert_end,horz_start:horz_end,:]
            A[:,h,w,:]=np.max(a_prev_slice, axis=(1,2))

    cache=(A_prev,s,f)
    return A,cache




In [ ]:
def flatten(A):
    return A.reshape(A.shape[0],-1).T

In [ ]:
def fc_forward(A_prev,W,b):
    Z=np.dot(W,A_prev)+b
    
    A=np.maximum(0,Z)

    linear_cache=A_prev,W,b
    activation_cache=Z
    cache=(linear_cache,activation_cache)
    
    return A,cache

In [ ]:
def softmax(A_prev,W,b):
    Z=np.dot(W,A_prev)+b
    shift_Z = Z - np.max(Z, axis=0, keepdims=True)
    exp_Z = np.exp(shift_Z)
    A=exp_Z / np.sum(exp_Z, axis=0, keepdims=True)

    linear_cache=(A_prev,W,b)
    activation_cache=Z
    cache=(linear_cache,activation_cache)

    return A,cache

In [ ]:
p=get_parameters()
h=get_hparameters()

def forward_prop(X,parameters,hparameters):
    caches=[]
    #parameters
    Wc1=parameters["Wc1"]
    bc1=parameters["bc1"]
    Wc2=parameters["Wc2"]
    bc2=parameters["bc2"]
    Wfc1=parameters["Wfc1"]
    bfc1=parameters["bfc1"]
    Wfc2=parameters["Wfc2"]
    bfc2=parameters["bfc2"]
    #hparameters
    sc1=hparameters["sc1"]
    pc1=hparameters["pc1"]
    sc2=hparameters["sc2"]
    pc2=hparameters["pc2"]
    sp1=hparameters["sp1"]
    fp1=hparameters["fp1"]
    sp2=hparameters["sp2"]
    fp2=hparameters["fp2"]


    Ac1,c=conv_forward(X,Wc1,bc1,sc1,pc1)
    caches.append(c)
    Ap1,c=pool_forward(Ac1,sp1,fp1)
    caches.append(c)
    Ac2,c=conv_forward(Ap1,Wc2,bc2,sc2,pc2)
    caches.append(c)
    Ap2,c=pool_forward(Ac2,sp2,fp2)
    caches.append(c)

    Ap2=flatten(Ap2)
    Afc1,c=fc_forward(Ap2,Wfc1,bfc1)
    caches.append(c)
    Afc2,c=softmax(Afc1,Wfc2,bfc2)
    caches.append(c)

    A_L=Afc2

    return A_L,caches

A_L,caches=forward_prop(X_train,p,h)
print (A_L.shape)




    

ValueError: shapes (128,16384) and (1024,1080) not aligned: 16384 (dim 1) != 1024 (dim 0)

cost

In [ ]:
A_L,caches=forward_prop(X_train,p,h)
def cost(A_L,Y):
    m=Y.shape[1]
    cost_val= -(1/m)*np.sum(Y*np.log(A_L+ 1e-8))
    print(cost_val.shape)
    return cost_val

print(cost(A_L,Y_train))
print(len(caches))



    

back propagation

In [ ]:
def fc_backprop(dA,cache,activation):
    (A_prev,W,b),Z=cache
    m=A_prev.shape[1]
    if activation=="softmax":
        dZ=dA
    
    else:
        dZ=np.array(dA,copy=True)
        dZ[Z<=0] =0

    dW=(1/m) * np.dot(dZ,A_prev.T)
    db=(1/m) * np.sum(dZ, axis=1 , keepdims=True)
    dA_prev=np.dot(W.T,dZ)

    return dA_prev,dW,db


In [ ]:
# 16 16 64
def unflatten(dA_flatten, shape_before_flatten):
    return dA_flatten.T.reshape(shape_before_flatten)

In [ ]:
def pool_backward(dA,cache):
    (A_prev,s,f)=cache
    m,n_H,n_W,n_C=dA.shape
    dA_prev=np.zeros_like(A_prev)

    for h in range(n_H):
        for w in range(n_W):
            v_start,h_start=s*h,s*w
            a_prev_slice=A_prev[:,v_start:v_start+f,h_start:h_start+f,:]
            mask=(a_prev_slice == np.max(a_prev_slice,axis=(1,2),keepdims=True))

            dA_prev[:,v_start:v_start+f,h_start:h_start+f,:]+=dA[:,h:h+1,w:w+1,:]*mask

    return dA_prev        

In [ ]:
def conv_backward(dA,cache):

    Z=cache[3]
    dZ=np.array(dA,copy=True)
    dZ[Z<=0]=0



    A_prev,W,b,Z,s,p=cache
    m,n_H,n_W,n_C=dZ.shape
    f,f,n_C_prev,n_C_filters=W.shape

    dA_prev=np.zeros_like(A_prev,dtype=np.float64)
    dW=np.zeros_like(W)
    db=np.sum(dZ,axis=(0,1,2),keepdims=True)

    A_prev_pad=zero_pad(A_prev,p)
    dA_prev_pad=zero_pad(dA_prev,p)

    for h in range(n_H):
        for w in range(n_W):
            v_start,h_start=s*h, s*w
            a_slice = A_prev_pad[:, v_start:v_start+f, h_start:h_start+f, :]
            dZ_at_pos = dZ[:, h, w, :]
            dW += np.dot(a_slice.reshape(m, -1).T, dZ_at_pos).reshape(f, f, n_C_prev, n_C)
            
            # Gradient for A_prev: (m, n_C) dot (n_C, f*f*n_C_prev)
            dA_prev_pad[:, v_start:v_start+f, h_start:h_start+f, :] += \
                np.dot(dZ_at_pos, W.reshape(-1, n_C_filters).T).reshape(m, f, f, n_C_prev)



    if p != 0:
        dA_prev = dA_prev_pad[:, p:-p, p:-p, :]
    else:
        dA_prev = dA_prev_pad
        
    return dA_prev, dW/m, db/m


In [ ]:
def back_prop(A_L,Y,caches):
    m=A_L.shape[1]
    grads={}

    dZ_fc2=A_L-Y
    dA_fc1,dWfc2,dbfc2 = fc_backprop(dZ_fc2,caches[5],activation="softmax")
    grads["Wfc2"]=dWfc2
    grads["bfc2"]=dbfc2

    dA_p2_flat,dWfc1,dbfc1=fc_backprop(dA_fc1,caches[4],activation="relu")
    grads["Wfc1"]=dWfc1
    grads["bfc1"]=dbfc1

    dA_p2=unflatten(dA_p2_flat, (m,16,16,64))
    dA_c2=pool_backward(dA_p2,caches[3])

    dA_p1,dWc2,dbc2=conv_backward(dA_c2,caches[2])
    grads["Wc2"]=dWc2
    grads["bc2"]=dbc2

    dAc1=pool_backward(dA_p1,caches[1])

    dA_input,dWc1,dbc1=conv_backward(dAc1,caches[0])
    grads["Wc1"]=dWc1
    grads["bc1"]=dbc1

    return grads


    




    
    

optimizer

In [ ]:
def optimizer(par,grad,learning_rate):

    par["Wc1"]-=grad["Wc1"]*learning_rate
    par["Wc2"]-=grad["Wc2"]*learning_rate
    par["bc1"]-=grad["bc1"]*learning_rate
    par["bc2"]-=grad["bc2"]*learning_rate
    par["Wfc1"]-=grad["Wfc1"]*learning_rate
    par["Wfc2"]-=grad["Wfc2"]*learning_rate
    par["bfc2"]-=grad["bfc2"]*learning_rate
    par["bfc1"]-=grad["bfc1"]*learning_rate


    return par
    
    

model

In [ ]:
def model(X,Y,no_iteration,learning_rate):

    parameters=get_parameters()
    hparameters=get_hparameters()

    for i in range(0,no_iteration):

        A_L,caches=forward_prop(X,parameters,hparameters)
        gradients=back_prop(A_L,Y,caches)
        parameters=optimizer(parameters,gradients,learning_rate)

        if i%10==0 :
            print("iteration =",i,"  " , cost(A_L,Y))


    return parameters






In [ ]:
def predict(X, parameters, hparameters):
    # Only need the forward pass
    A_L, _ = forward_prop(X, parameters, hparameters)
    
    # Get the index of the max probability for each example
    predictions = np.argmax(A_L, axis=0)
    return predictions


def get_accuracy(predictions, Y):
    # Y is one-hot (6, m), so we need the true indices
    labels = np.argmax(Y, axis=0)
    return np.mean(predictions == labels) * 100

In [ ]:
h=get_hparameters()
final_params = model(X_train, Y_train, no_iteration=1500, learning_rate=0.01)



In [ ]:

predictions=predict(X_train,final_params,h)

print(get_accuracy(predictions,Y_train))